# NeuroGolf 2026 - Minimal Neural Networks for ARC-AGI

Train the smallest possible ONNX neural networks to solve ARC-AGI tasks.

**Strategy:**
1. Try handcrafted solutions first (color remaps)
2. Train progressively larger CNNs until one solves the task
3. Export smallest working network to ONNX
4. Create submission.zip

In [ ]:
import json
import math
import os
import time
import zipfile
from pathlib import Path

import numpy as np
import onnx
import onnx.helper
import onnx.numpy_helper
import onnxruntime
import torch
import torch.nn as nn
import torch.optim as optim

# Paths
DATA_DIR = Path("/kaggle/input/competitions/neurogolf-2026")
OUTPUT_DIR = Path("/kaggle/working")
SUBMISSION_DIR = OUTPUT_DIR / "submission"
SUBMISSION_DIR.mkdir(exist_ok=True)

# Constants
CHANNELS = 10
HEIGHT = WIDTH = 30
GRID_SHAPE = [1, CHANNELS, HEIGHT, WIDTH]
IR_VERSION = 10
OPSET_IMPORTS = [onnx.helper.make_opsetid("", 10)]
DATA_TYPE = onnx.TensorProto.FLOAT

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================
# Data Loading
# ============================================================

def load_task(task_num):
    with open(DATA_DIR / f"task{task_num:03d}.json") as f:
        return json.load(f)


def grid_to_tensor(grid):
    t = torch.zeros(1, CHANNELS, HEIGHT, WIDTH, dtype=torch.float32)
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            t[0, color, r, c] = 1.0
    return t


def grid_to_numpy(grid):
    t = np.zeros((1, CHANNELS, HEIGHT, WIDTH), dtype=np.float32)
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            t[0, color, r, c] = 1.0
    return t


def examples_to_tensors(examples):
    inputs = [grid_to_tensor(ex["input"]) for ex in examples]
    outputs = [grid_to_tensor(ex["output"]) for ex in examples]
    return torch.cat(inputs, dim=0), torch.cat(outputs, dim=0)


def check_accuracy(model, inputs, targets):
    model.eval()
    with torch.no_grad():
        preds = model(inputs)
        binary_preds = (preds > 0.0).float()
        match = torch.all(binary_preds == targets, dim=(1, 2, 3))
        return match.float().mean().item(), match.sum().item(), len(match)

In [ ]:
# ============================================================
# Network Architectures
# ============================================================

class SingleConv(nn.Module):
    def __init__(self, kernel_size=3):
        super().__init__()
        pad = kernel_size // 2
        self.conv = nn.Conv2d(CHANNELS, CHANNELS, kernel_size, padding=pad, bias=False)

    def forward(self, x):
        return self.conv(x)


class BottleneckConv(nn.Module):
    def __init__(self, hidden=4, kernel_size=3):
        super().__init__()
        pad = kernel_size // 2
        self.net = nn.Sequential(
            nn.Conv2d(CHANNELS, hidden, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(hidden, hidden, kernel_size, padding=pad, bias=False),
            nn.ReLU(),
            nn.Conv2d(hidden, CHANNELS, 1, bias=False),
        )

    def forward(self, x):
        return self.net(x)


class ResBlock(nn.Module):
    def __init__(self, channels, kernel_size=3):
        super().__init__()
        pad = kernel_size // 2
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size, padding=pad, bias=False),
            nn.ReLU(),
            nn.Conv2d(channels, channels, kernel_size, padding=pad, bias=False),
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(x + self.net(x))


class ResidualConv(nn.Module):
    def __init__(self, hidden=8, kernel_size=3, num_blocks=2):
        super().__init__()
        pad = kernel_size // 2
        layers = [nn.Conv2d(CHANNELS, hidden, kernel_size, padding=pad, bias=False), nn.ReLU()]
        for _ in range(num_blocks - 1):
            layers += [nn.Conv2d(hidden, hidden, kernel_size, padding=pad, bias=False), nn.ReLU()]
        layers.append(nn.Conv2d(hidden, CHANNELS, kernel_size, padding=pad, bias=False))
        self.body = nn.Sequential(*layers)

    def forward(self, x):
        return x + self.body(x)


class DilatedConv(nn.Module):
    def __init__(self, hidden=8, kernel_size=3):
        super().__init__()
        pad1 = kernel_size // 2
        pad2 = (kernel_size // 2) * 2
        pad4 = (kernel_size // 2) * 4
        self.net = nn.Sequential(
            nn.Conv2d(CHANNELS, hidden, kernel_size, padding=pad1, dilation=1, bias=False),
            nn.ReLU(),
            nn.Conv2d(hidden, hidden, kernel_size, padding=pad2, dilation=2, bias=False),
            nn.ReLU(),
            nn.Conv2d(hidden, hidden, kernel_size, padding=pad4, dilation=4, bias=False),
            nn.ReLU(),
            nn.Conv2d(hidden, CHANNELS, 1, bias=False),
        )

    def forward(self, x):
        return self.net(x)


class MultiScaleConv(nn.Module):
    def __init__(self, hidden=8):
        super().__init__()
        self.local = nn.Sequential(
            nn.Conv2d(CHANNELS, hidden, 3, padding=1, bias=False),
            nn.ReLU(),
            nn.Conv2d(hidden, hidden, 3, padding=1, bias=False),
            nn.ReLU(),
        )
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.global_fc = nn.Sequential(
            nn.Conv2d(CHANNELS, hidden, 1, bias=False),
            nn.ReLU(),
        )
        self.combine = nn.Conv2d(hidden * 2, CHANNELS, 1, bias=False)

    def forward(self, x):
        local_feat = self.local(x)
        global_feat = self.global_fc(self.global_pool(x))
        global_feat = global_feat.expand_as(local_feat)
        combined = torch.cat([local_feat, global_feat], dim=1)
        return self.combine(combined)


class DeepResidual(nn.Module):
    def __init__(self, hidden=16, kernel_size=3, num_blocks=3):
        super().__init__()
        self.input_proj = nn.Conv2d(CHANNELS, hidden, 1, bias=False)
        self.blocks = nn.Sequential(*[ResBlock(hidden, kernel_size) for _ in range(num_blocks)])
        self.output_proj = nn.Conv2d(hidden, CHANNELS, 1, bias=False)

    def forward(self, x):
        h = self.input_proj(x)
        h = self.blocks(h)
        return self.output_proj(h) + x


ARCH_CONFIGS = [
    ("conv1x1", lambda: SingleConv(kernel_size=1)),
    ("conv3x3", lambda: SingleConv(kernel_size=3)),
    ("conv5x5", lambda: SingleConv(kernel_size=5)),
    ("conv7x7", lambda: SingleConv(kernel_size=7)),
    ("bneck_h4_k3", lambda: BottleneckConv(hidden=4, kernel_size=3)),
    ("bneck_h4_k5", lambda: BottleneckConv(hidden=4, kernel_size=5)),
    ("bneck_h8_k3", lambda: BottleneckConv(hidden=8, kernel_size=3)),
    ("bneck_h8_k5", lambda: BottleneckConv(hidden=8, kernel_size=5)),
    ("res_h8_b2", lambda: ResidualConv(hidden=8, num_blocks=2)),
    ("res_h16_b2", lambda: ResidualConv(hidden=16, num_blocks=2)),
    ("res_h8_b3", lambda: ResidualConv(hidden=8, num_blocks=3)),
    ("dilated_h8", lambda: DilatedConv(hidden=8)),
    ("dilated_h16", lambda: DilatedConv(hidden=16)),
    ("multiscale_h8", lambda: MultiScaleConv(hidden=8)),
    ("multiscale_h16", lambda: MultiScaleConv(hidden=16)),
    ("deepres_h16_b3", lambda: DeepResidual(hidden=16, num_blocks=3)),
    ("deepres_h32_b3", lambda: DeepResidual(hidden=32, num_blocks=3)),
    ("deepres_h16_b5", lambda: DeepResidual(hidden=16, num_blocks=5)),
]

In [ ]:
# ============================================================
# Training
# ============================================================

def compute_loss(preds, targets):
    bce = nn.functional.binary_cross_entropy_with_logits(preds, targets)
    mse = nn.functional.mse_loss(preds, targets * 2 - 1)
    return bce + 0.1 * mse


def train_model(model, train_inputs, train_targets, max_epochs=1500, lr=0.003,
                patience=300, batch_size=64, early_fail_epochs=150):
    model = model.to(DEVICE)
    train_inputs = train_inputs.to(DEVICE)
    train_targets = train_targets.to(DEVICE)

    n = train_inputs.shape[0]
    best_correct = 0
    best_acc = 0.0
    best_state = None
    no_improve = 0

    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-6)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=400, T_mult=2)

    for epoch in range(max_epochs):
        model.train()
        if n <= batch_size:
            optimizer.zero_grad()
            loss = compute_loss(model(train_inputs), train_targets)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        else:
            perm = torch.randperm(n, device=DEVICE)
            for i in range(0, n, batch_size):
                idx = perm[i:i+batch_size]
                optimizer.zero_grad()
                loss = compute_loss(model(train_inputs[idx]), train_targets[idx])
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
        scheduler.step()

        if (epoch + 1) % 25 == 0:
            acc, correct, total = check_accuracy(model, train_inputs, train_targets)
            if (epoch + 1) >= early_fail_epochs and best_correct == 0:
                return False, 0.0, epoch + 1, None
            if correct > best_correct:
                best_correct = correct
                best_acc = acc
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 25
            if acc == 1.0:
                return True, acc, epoch + 1, best_state
            if no_improve >= patience:
                break

    if best_state:
        model.load_state_dict(best_state)
        model = model.to(DEVICE)
    return best_acc == 1.0, best_acc, epoch + 1, best_state

In [ ]:
# ============================================================
# ONNX Export & Validation
# ============================================================

def export_to_onnx(model, task_num):
    model = model.cpu().eval()
    filepath = str(SUBMISSION_DIR / f"task{task_num:03d}.onnx")
    dummy = torch.randn(1, CHANNELS, HEIGHT, WIDTH)
    # Use legacy TorchScript exporter for reliable opset 10 support
    export_kwargs = dict(
        opset_version=10,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes=None,
    )
    # PyTorch >= 2.9 defaults to dynamo exporter which can't target opset 10
    import inspect
    if "dynamo" in inspect.signature(torch.onnx.export).parameters:
        export_kwargs["dynamo"] = False
    torch.onnx.export(model, dummy, filepath, **export_kwargs)
    return filepath


def validate_onnx(filepath, examples):
    fpath = Path(filepath)
    if not fpath.exists() or fpath.stat().st_size > 1.44 * 1024 * 1024:
        return False, 0.0, "file issue"
    try:
        session = onnxruntime.InferenceSession(filepath)
    except Exception as e:
        return False, 0.0, str(e)

    all_ex = examples.get("train", []) + examples.get("test", []) + examples.get("arc-gen", [])
    correct = 0
    for ex in all_ex:
        inp = grid_to_numpy(ex["input"])
        expected = grid_to_numpy(ex["output"])
        try:
            result = session.run(["output"], {"input": inp})
            if np.array_equal((result[0] > 0.0).astype(np.float32), expected):
                correct += 1
        except Exception:
            pass
    if correct < len(all_ex):
        return False, 0.0, f"{correct}/{len(all_ex)}"

    try:
        import onnx_tool
        m = onnx_tool.loadmodel(filepath, {'verbose': False})
        g = m.graph
        g.graph_reorder_nodes()
        g.shape_infer(None)
        g.profile()
        macs, memory, params = int(sum(g.macs)), int(g.memory), int(g.params)
        score = max(1.0, 25.0 - math.log(macs + memory + params))
        return True, score, f"score={score:.2f}"
    except Exception:
        return True, 1.0, "scored=1.0"

In [ ]:
# ============================================================
# Handcrafted Solutions
# ============================================================

def solve_color_remap(task_num, examples):
    """Try pure color remap (1x1 conv)."""
    all_ex = examples["train"] + examples["test"]
    cmap = {}
    for ex in all_ex:
        inp, out = ex["input"], ex["output"]
        if len(inp) != len(out) or len(inp[0]) != len(out[0]):
            return None
        for r in range(len(inp)):
            for c in range(len(inp[0])):
                ic, oc = inp[r][c], out[r][c]
                if ic in cmap and cmap[ic] != oc:
                    return None
                cmap[ic] = oc

    weights = np.zeros((CHANNELS, CHANNELS, 1, 1), dtype=np.float32)
    for ic, oc in cmap.items():
        weights[oc, ic, 0, 0] = 1.0
    for c in range(CHANNELS):
        if c not in cmap:
            weights[c, c, 0, 0] = 1.0

    x = onnx.helper.make_tensor_value_info("input", DATA_TYPE, GRID_SHAPE)
    y = onnx.helper.make_tensor_value_info("output", DATA_TYPE, GRID_SHAPE)
    w = onnx.numpy_helper.from_array(weights, name="W")
    node = onnx.helper.make_node("Conv", ["input", "W"], ["output"],
                                  kernel_shape=[1, 1], pads=[0, 0, 0, 0])
    graph = onnx.helper.make_graph([node], "graph", [x], [y], [w])
    model = onnx.helper.make_model(graph, ir_version=IR_VERSION, opset_imports=OPSET_IMPORTS)

    filepath = str(SUBMISSION_DIR / f"task{task_num:03d}.onnx")
    onnx.save(model, filepath)
    passed, score, details = validate_onnx(filepath, examples)
    if passed:
        return filepath
    os.remove(filepath)
    return None

In [ ]:
# ============================================================
# Main Solver
# ============================================================

def solve_task(task_num, verbose=False):
    examples = load_task(task_num)
    result = {"task": task_num, "solved": False, "score": 0.0, "tier": None}

    all_ex = examples["train"] + examples["test"] + examples.get("arc-gen", [])
    train_inputs, train_targets = examples_to_tensors(all_ex)
    n = len(all_ex)

    # 1. Try handcrafted
    fp = solve_color_remap(task_num, examples)
    if fp:
        _, score, details = validate_onnx(fp, examples)
        result.update(solved=True, score=score, tier="handcrafted")
        if verbose:
            print(f"  Task {task_num:3d}: SOLVED (handcrafted) {details}")
        return result

    # 2. Try learned architectures
    for arch_name, arch_fn in ARCH_CONFIGS:
        model = arch_fn()
        pc = sum(p.numel() for p in model.parameters())
        me = 2000 if pc < 1000 else (1500 if pc < 10000 else 1000)
        bs = min(64, n)

        ok, acc, ep, state = train_model(
            model, train_inputs, train_targets,
            max_epochs=me, lr=0.003, patience=300,
            batch_size=bs, early_fail_epochs=150
        )

        if ok and state:
            model.load_state_dict(state)
            fp = export_to_onnx(model, task_num)
            passed, score, details = validate_onnx(fp, examples)
            if passed:
                result.update(solved=True, score=score, tier=arch_name)
                if verbose:
                    print(f"  Task {task_num:3d}: SOLVED ({arch_name}) {details}")
                return result
            os.remove(fp)

            # Retry with different lr
            m2 = arch_fn()
            ok2, _, _, st2 = train_model(
                m2, train_inputs, train_targets,
                max_epochs=me, lr=0.01, patience=300,
                batch_size=bs, early_fail_epochs=150
            )
            if ok2 and st2:
                m2.load_state_dict(st2)
                fp = export_to_onnx(m2, task_num)
                passed, score, details = validate_onnx(fp, examples)
                if passed:
                    result.update(solved=True, score=score, tier=arch_name)
                    if verbose:
                        print(f"  Task {task_num:3d}: SOLVED ({arch_name},retry) {details}")
                    return result
                os.remove(fp)

    if verbose:
        print(f"  Task {task_num:3d}: UNSOLVED")
    return result

In [ ]:
# ============================================================
# Run All Tasks
# ============================================================

results = []
total_score = 0.0
solved = 0
t_start = time.time()

for task_num in range(1, 401):
    t0 = time.time()
    r = solve_task(task_num, verbose=True)
    dt = time.time() - t0
    results.append(r)
    if r["solved"]:
        solved += 1
        total_score += r["score"]
    tag = "OK" if r["solved"] else "--"
    print(f"  [{dt:5.1f}s] Task {task_num:3d} {tag} | {solved} solved, {total_score:.1f} pts")

elapsed = time.time() - t_start
print(f"\n{'='*60}")
print(f"Solved: {solved}/400")
print(f"Score:  {total_score:.1f}")
print(f"Time:   {elapsed:.0f}s ({elapsed/60:.1f}m)")
print(f"{'='*60}")

In [ ]:
# ============================================================
# Create Submission
# ============================================================

zip_path = OUTPUT_DIR / "submission.zip"
onnx_files = sorted(SUBMISSION_DIR.glob("task*.onnx"))
print(f"Found {len(onnx_files)} ONNX files")

if onnx_files:
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in onnx_files:
            zf.write(f, f.name)
    print(f"Created {zip_path} ({zip_path.stat().st_size / 1024:.0f} KB)")

# Save results
with open(OUTPUT_DIR / "results.json", "w") as f:
    json.dump(results, f, indent=2)
print("Done!")